# GPU T4 : données persistées, mesure de débit, premières epochs
Ouvrir un **nouveau notebook GPU**, activer une Tesla T4 et Internet pour les sauvegardes.
Coller les vrais reçus CPU ; aucun score expérimental n'est prérempli.
Les cellules coûteuses sont désactivées par défaut. Le total reste fixé à 30 epochs,
avec un premier arrêt après deux epochs pour mesurer le coût réel et reprendre ensuite.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
CODE_SOURCE = Path('/kaggle/input/datasets/nnnnbhss/geofirex-code-v3/GeoFire-X-2026')  # adapter au dataset de code v3
CODE = Path('/kaggle/working/GeoFire-X-2026-v3')
assert CODE_SOURCE.is_dir(), 'Renseigner le chemin du code v3 attaché au notebook'
if not CODE.exists():
    shutil.copytree(CODE_SOURCE, CODE)
manifest = json.loads((CODE_SOURCE/'code_manifest.json').read_text())
assert all(hashlib.sha256((CODE/name).read_bytes()).hexdigest() == digest for name, digest in manifest.items()), 'Code local différent du paquet v3'
os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kagglehub>=1.0.3,<2'], check=True)
sys.path.insert(0, str(CODE/'src'))
from kagglehub.auth import get_username
assert get_username() == 'nnnnbhss', 'Compte Kaggle authentifié inattendu'
print('Compte vérifié :', get_username())


## Vérifier les lots CPU depuis cette nouvelle session et construire le manifeste cumulatif


In [ ]:
BATCH_RECEIPTS_JSON = ''  # coller la liste JSON des reçus train2018 et val2020 (et 2019 si disponible)
from geofirex.kaggle_persistence import verify_fresh_session, session_id, KaggleBackend
from geofirex.cloud_corpus import prepare_mounted
from geofirex.corpus import json_new
assert BATCH_RECEIPTS_JSON.strip(), 'Les vrais reçus CPU sont requis'
receipts = json.loads(BATCH_RECEIPTS_JSON)
assert len(receipts) >= 2
backend = KaggleBackend()
roots = []
for receipt in receipts:
    certificate = verify_fresh_session(receipt, session=session_id(), backend=backend)
    root = backend.mount(receipt['versioned_handle'])
    if root not in roots:
        roots.append(root)
    print(certificate['status'], receipt['versioned_handle'])
manual_root = Path('/kaggle/input/datasets/nnnnbhss/geofire-x-wsts-batch001')
if manual_root.is_dir() and manual_root not in roots:
    from geofirex.kaggle_persistence import inspect_dataset
    assert inspect_dataset('nnnnbhss/geofire-x-wsts-batch001')['state'] == 'EXISTING_PRIVATE'
    roots.append(manual_root)  # joindre batch001 dans Add Input pour l’inclure
index_path = Path('data/manifests/real/corpus_zip_index_v1.json')
index = json.loads(index_path.read_text())
prepared = Path('/kaggle/working/prepared-v3')
assert not prepared.exists(), 'Utiliser une nouvelle sortie ou reprendre le manifeste existant identique'
report = prepare_mounted(index, roots, prepared)
print(json.dumps(report, indent=2))


## Configurations comparables, quotas de checkpoints et GPU réel


In [ ]:
PRIVATE_CHECKPOINT_FREE_GIB = None  # disponible réel du compte, APRÈS acquisition
APPROVED_CHECKPOINT_UPLOAD_GIB = None  # budget total de payload pour dix runs, à approuver
assert PRIVATE_CHECKPOINT_FREE_GIB and APPROVED_CHECKPOINT_UPLOAD_GIB
from scripts.kaggle_campaign import configure
import torch
assert torch.cuda.is_available(), 'CUDA indisponible ; ne pas remplacer le build PyTorch CUDA'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory)
campaign = Path('/kaggle/working/campaign-v3')
configs = configure(prepared, index_path, campaign, 'nnnnbhss/geofirex-pilot-v3',
    int(APPROVED_CHECKPOINT_UPLOAD_GIB*1024**3), int(PRIVATE_CHECKPOINT_FREE_GIB*1024**3))
print([(c['model'], c['seed'], c['planned_optimizer_updates']) for c in configs])


## Première mesure de débit GPU — modèle jeté, aucune métrique scientifique
100 pas mesurés + 10 pas d'échauffement. Lire les temps et la VRAM avant d'autoriser les epochs.


In [ ]:
RUN_THROUGHPUT = False
MODEL = 'B1'  # puis B2/B3 pour mesurer leur coût ; B1 utilise spatial_cnn_v2 explicite
config_path = campaign/f'{MODEL}_s42.yaml'
if RUN_THROUGHPUT:
    subprocess.run([sys.executable, '-m', 'scripts.measure_gpu_throughput', '--config', str(config_path),
        '--steps', '100', '--warmup', '10', '--out', f'/kaggle/working/throughput-{MODEL}.json'], check=True)
    print(Path(f'/kaggle/working/throughput-{MODEL}.json').read_text())


## B0 une fois, puis deux vraies epochs par modèle entraînable
Deux epochs représentent 1 024 updates planifiées par modèle, avant les pas éventuellement sautés par AMP. B0 n'a pas d'optimiseur. Chaque epoch et la fin de session produisent une sauvegarde privée vérifiée côté producteur.


In [ ]:
RUN_B0 = False
if RUN_B0:
    subprocess.run([sys.executable, 'scripts/run_benchmark.py', 'train', '--config', str(campaign/'B0_s42.yaml')], check=True)


In [ ]:
RUN_TWO_EPOCHS = False
if RUN_TWO_EPOCHS:
    subprocess.run([sys.executable, 'scripts/run_benchmark.py', 'train', '--config', str(config_path),
        '--stop-after-epochs', '2'], check=True)


## Sauvegardes et reprise dans une autre session
Conserver le reçu `backup_0002_final.json` hors de la VM. Dans une nouvelle session, vérifier ce reçu avec `verify_fresh_session`, puis monter sa version Kaggle. Recréer le même manifeste et les mêmes configurations à partir du même paquet de code. `stop-after-epochs` ne modifie pas l'identité scientifique du run. Une session partielle porte l'état `PAUSED_AT_EPOCH_BOUNDARY` et ne peut pas être classée comme un run terminé.


In [ ]:
print('Reçus de sauvegarde à conserver :')
for path in Path('/kaggle/working/geofirex-runs').glob('*/backup_*_final.json'):
    print(path)
    print(path.read_text())


In [ ]:
RUN_RESUME = False
CHECKPOINT_RECEIPT_JSON = ''  # reçu d'une AUTRE session, à vérifier ici
if RUN_RESUME:
    receipt = json.loads(CHECKPOINT_RECEIPT_JSON)
    print(verify_fresh_session(receipt, session=session_id(), backend=backend))
    snapshot = backend.mount(receipt['versioned_handle'])
    checkpoints = sorted((snapshot/'run').glob('epoch_*.pt'))
    assert checkpoints, 'Aucun checkpoint complet dans le snapshot vérifié'
    subprocess.run([sys.executable, 'scripts/run_benchmark.py', 'train', '--config', str(config_path),
        '--resume', str(checkpoints[-1]), '--stop-after-epochs', '2'], check=True)


## Étape Big Data suivante — après les lots, dans CPU
Exporter les métadonnées réelles montées en Parquet, puis lancer Spark avec au moins 30 événements et 500 observations. Une exécution `local[2]` est un traitement Spark local, pas un cluster. PySpark/JVM ne sont pas installés ni validés par les tests locaux de ce notebook.


In [ ]:
import shlex
metadata = '/kaggle/working/metadata-pilot-v3'
print(shlex.join([sys.executable, '-m', 'scripts.export_observation_metadata', '--manifest', str(prepared/'manifest.csv'),
    '--out-dir', metadata, '--data-roots', *map(str, roots)]))
print(shlex.join([sys.executable, '-m', 'scripts.spark_metadata', '--input', metadata+'/observations.parquet',
    '--output', '/kaggle/working/spark-pilot-v3', '--master', 'local[2]', '--min-events', '30',
    '--min-observations', '500', '--out-report', '/kaggle/working/spark-pilot-v3.json']))
